# GateKeep · notebook 11 · v2 evaluation (plan Tasks 9-11)

**Attach:** `gatekeep_data` (the new version) and notebook 10's output. **Settings:** GPU T4, Internet On, secret `OLLAMA_API_KEY` (the new key).
Run with **Save Version → Save & Run All**, then download `gatekeep_backup.zip` from Output.

## 1 · code, packages, key

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.isdir("GateKeep"):
    subprocess.run(["git", "-C", "GateKeep", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", "v2", "https://github.com/RishabhCodezZz/GateKeep.git"], check=True)
os.chdir("GateKeep")
os.environ["PYTHONPATH"] = "src"   # so `!python -m gatekeep.cli` finds the package
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" safetensors huggingface_hub scipy accelerate faiss-cpu sentence-transformers pymupdf pytest openai docling

In [ ]:
!nvidia-smi -L

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["OLLAMA_API_KEY"] = UserSecretsClient().get_secret("OLLAMA_API_KEY")

## 2 · restore (cache and data from the backup, models from notebook 10's output)

In [ ]:
import sys
sys.path.insert(0, "src")
from gatekeep.restore import restore

COLD_START_OK = False   # set True only if you really want to rebuild everything (costs ~11% of the monthly quota)
where = restore("/kaggle/input", ".")
if where:
    print("restored from", where)
    subprocess.run(["python", "-m", "gatekeep.cli", "purge_empty"], check=True)  # drop empty replies cached before the thinking fix
elif not COLD_START_OK:
    raise SystemExit("STOPPED: no backup found in /kaggle/input. Attach your backup (the dataset must contain the folders "
                     "data/, cache/, results/ or a gatekeep_backup.zip), or set COLD_START_OK = True to rebuild from scratch.")
else:
    print("no backup: cold start, the book is re-parsed and every model call is repeated")
for name in ("review.csv", "handwritten.csv"):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found and not os.path.exists(name):
        shutil.copy(found[0], name)
    print(name, "->", "found" if os.path.exists(name) else "MISSING (add it to your dataset)")

In [ ]:
missing = [g for g in ("route", "grade", "grounded", "sufficient") if not os.path.exists(f"models/{g}/model.safetensors")]
assert not missing, (f"fine-tuned weights missing for {missing}. Attach notebook 10's OUTPUT as an input "
                     "(Add Input -> Notebook Outputs); it only has them if it was saved with Save & Run All. Otherwise re-run notebook 10.")
print("fine-tuned models found:", sorted(os.listdir("models")))

## 3 · calibrate on the dev split

In [ ]:
!python -m gatekeep.cli calibrate

## 4 · per-gate results

In [ ]:
!python -m gatekeep.cli evalgates llm,sklearn,laya-zero,laya-ft
!cat results/tau.json

## 5 · V1, V2, V3 on the test set

In [ ]:
!python -m gatekeep.cli variants V1,V2,V3

## 6 · the 80 hand-written questions

In [ ]:
!python -m gatekeep.cli variants V1,V3 "" results/tau.json 0 hw_ data/handwritten.jsonl

## 7 · breakdown, verdict and plots

In [ ]:
!python -m gatekeep.cli breakdown
!python -m gatekeep.cli report

## 8 · scikit-learn docs (out of distribution)

In [ ]:
if not os.path.isdir("data_book"): shutil.move("data", "data_book"); os.mkdir("data")
try:
    for cmd in (["python", "scripts/fetch_sklearn_docs.py"],
                ["python", "-m", "gatekeep.cli", "prepare_dir", "data/sk"],
                ["python", "-m", "gatekeep.cli", "questions", "0", "0", "200"],
                ["python", "-m", "gatekeep.cli", "variants", "V1,V3", "", "results/tau.json", "0", "ood_"]):
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print(r.stdout[-3000:])
        assert r.returncode == 0, cmd
finally:
    shutil.rmtree("data"); shutil.move("data_book", "data")   # always put the book's data back

## 9 · back up

In [ ]:
!cd /kaggle/working/GateKeep && zip -rq /kaggle/working/gatekeep_backup.zip data cache results models/ckpts.json models/*/calibration.json && ls -lh /kaggle/working/gatekeep_backup.zip